# 03 — Transform Invoices to Silver

## Purpose

Transform validated Bronze invoice events into a clean, current-state Silver invoice table.

This notebook will standardize invoice attributes, validate customer and subscription ownership, reconcile invoice financial amounts, and persist the result through an idempotent Delta merge.

## Sources

- `workspace.revenue_leakage_bronze.invoice_events`
- `workspace.revenue_leakage_silver.customers`
- `workspace.revenue_leakage_silver.subscriptions`

## Target

- `workspace.revenue_leakage_silver.invoices`

## 1. Load and Inspect Invoice Sources

Load the Bronze invoice events and the required Silver reference tables, then inspect the exact invoice schema and source-event distribution.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_INVOICES_TABLE = (
    "workspace.revenue_leakage_bronze.invoice_events"
)

SILVER_CUSTOMERS_TABLE = (
    "workspace.revenue_leakage_silver.customers"
)

SILVER_SUBSCRIPTIONS_TABLE = (
    "workspace.revenue_leakage_silver.subscriptions"
)

SILVER_INVOICES_TABLE = (
    "workspace.revenue_leakage_silver.invoices"
)

bronze_invoice_events_df = spark.table(
    BRONZE_INVOICES_TABLE
)

silver_customer_reference_df = spark.table(
    SILVER_CUSTOMERS_TABLE
)

silver_subscription_reference_df = spark.table(
    SILVER_SUBSCRIPTIONS_TABLE
)

print(
    f"Bronze invoice events: "
    f"{bronze_invoice_events_df.count():,}"
)

print(
    f"Silver customer references: "
    f"{silver_customer_reference_df.count():,}"
)

print(
    f"Silver subscription references: "
    f"{silver_subscription_reference_df.count():,}"
)

bronze_invoice_events_df.printSchema()

display(
    bronze_invoice_events_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

## 2. Standardize and Resolve Invoice Events

Standardize invoice attributes, remove exact duplicate events, and resolve the latest event for each invoice.

Each invoice must reference an existing Silver subscription, and its `customer_id` must match the customer who owns that subscription.

In [0]:
invoice_cdc_window = (
    Window
    .partitionBy("invoice_id")
    .orderBy(
        F.col("event_timestamp").desc(),
        F.col(
            "_source_file_modification_time"
        ).desc(),
        F.col("_ingested_at").desc(),
        F.col("_record_hash").desc(),
    )
)

standardized_invoice_events_df = (
    bronze_invoice_events_df
    .dropDuplicates(["_record_hash"])
    .withColumn(
        "invoice_id",
        F.trim(F.col("invoice_id")),
    )
    .withColumn(
        "subscription_id",
        F.trim(F.col("subscription_id")),
    )
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id")),
    )
    .withColumn(
        "billing_frequency",
        F.initcap(
            F.trim(F.col("billing_frequency"))
        ),
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency"))),
    )
    .withColumn(
        "invoice_status",
        F.initcap(
            F.trim(F.col("invoice_status"))
        ),
    )
    .withColumn(
        "operation",
        F.upper(F.trim(F.col("operation"))),
    )
)

latest_invoice_events_df = (
    standardized_invoice_events_df
    .withColumn(
        "_cdc_rank",
        F.row_number().over(
            invoice_cdc_window
        ),
    )
    .filter(F.col("_cdc_rank") == 1)
    .drop("_cdc_rank")
)

active_invoice_states_df = (
    latest_invoice_events_df
    .filter(F.col("operation") != "DELETE")
)

subscription_ownership_df = (
    silver_subscription_reference_df
    .select(
        "subscription_id",
        F.col("customer_id").alias(
            "_reference_customer_id"
        ),
    )
    .dropDuplicates(["subscription_id"])
)

invoice_reference_check_df = (
    active_invoice_states_df
    .join(
        subscription_ownership_df,
        on="subscription_id",
        how="left",
    )
)

orphan_invoice_states_df = (
    invoice_reference_check_df
    .filter(
        F.col("_reference_customer_id").isNull()
    )
)

ownership_mismatch_invoice_states_df = (
    invoice_reference_check_df
    .filter(
        F.col("_reference_customer_id").isNotNull()
        & (
            F.col("customer_id")
            != F.col("_reference_customer_id")
        )
    )
)

silver_invoices_df = (
    invoice_reference_check_df
    .filter(
        F.col("_reference_customer_id").isNotNull()
        & (
            F.col("customer_id")
            == F.col("_reference_customer_id")
        )
    )
    .select(
        "invoice_id",
        "subscription_id",
        "customer_id",
        "billing_period_start",
        "billing_period_end",
        "invoice_date",
        "due_date",
        "billing_frequency",
        "currency",
        "list_price_amount",
        "discount_percentage",
        "discount_amount",
        "net_subscription_amount",
        "overage_amount",
        "subtotal_amount",
        "tax_rate",
        "tax_amount",
        "invoice_total_amount",
        "amount_paid",
        "outstanding_amount",
        "voided_amount",
        "invoice_status",
        "payment_terms_days",
        F.col("operation").alias(
            "last_operation"
        ),
        F.col("event_timestamp").alias(
            "last_event_timestamp"
        ),
        "snapshot_date",
        "_source_system",
        "_source_entity",
        "_source_file_path",
        "_record_hash",
        F.current_timestamp().alias(
            "_silver_processed_at"
        ),
    )
)

print(
    f"Bronze invoice events: "
    f"{bronze_invoice_events_df.count():,}"
)

print(
    f"Distinct Bronze events: "
    f"{standardized_invoice_events_df.count():,}"
)

print(
    f"Latest invoice states: "
    f"{latest_invoice_events_df.count():,}"
)

print(
    f"Invoices without a Silver subscription: "
    f"{orphan_invoice_states_df.count():,}"
)

print(
    f"Subscription ownership mismatches: "
    f"{ownership_mismatch_invoice_states_df.count():,}"
)

print(
    f"Eligible Silver invoices: "
    f"{silver_invoices_df.count():,}"
)

display(
    silver_invoices_df
    .groupBy("invoice_status")
    .count()
    .orderBy("invoice_status")
)

## 3. Validate the Current Silver Invoice State

Validate invoice uniqueness, subscription ownership, business domains, billing dates, payment terms, financial calculations, and status-based balance allocation before persistence.

Invoices without a current Silver subscription are measured separately and excluded from the current-state dataset.

In [0]:
EXPECTED_SILVER_INVOICE_COUNT = 26_699
EXPECTED_EXCLUDED_INVOICE_COUNT = 226

ALLOWED_INVOICE_STATUSES = [
    "Open",
    "Paid",
    "Past Due",
    "Voided",
]

ALLOWED_INVOICE_BILLING_FREQUENCIES = [
    "Annual",
    "Monthly",
]

ALLOWED_INVOICE_CURRENCIES = [
    "USD",
]

required_invoice_columns = [
    "invoice_id",
    "subscription_id",
    "customer_id",
    "billing_period_start",
    "billing_period_end",
    "invoice_date",
    "due_date",
    "billing_frequency",
    "currency",
    "list_price_amount",
    "discount_percentage",
    "discount_amount",
    "net_subscription_amount",
    "overage_amount",
    "subtotal_amount",
    "tax_rate",
    "tax_amount",
    "invoice_total_amount",
    "amount_paid",
    "outstanding_amount",
    "voided_amount",
    "invoice_status",
    "payment_terms_days",
    "last_event_timestamp",
    "snapshot_date",
]

required_invoice_field_is_missing = None

for column_name in required_invoice_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == ""
        )
    )

    required_invoice_field_is_missing = (
        missing_condition
        if required_invoice_field_is_missing
        is None
        else required_invoice_field_is_missing
        | missing_condition
    )

invalid_invoice_status_condition = (
    ~F.col("invoice_status").isin(
        ALLOWED_INVOICE_STATUSES
    )
)

invalid_invoice_frequency_condition = (
    ~F.col("billing_frequency").isin(
        ALLOWED_INVOICE_BILLING_FREQUENCIES
    )
)

invalid_invoice_currency_condition = (
    ~F.col("currency").isin(
        ALLOWED_INVOICE_CURRENCIES
    )
)

invalid_invoice_operation_condition = (
    ~F.col("last_operation").isin(
        "INSERT",
        "UPDATE",
    )
)

invalid_invoice_date_condition = (
    (
        F.col("billing_period_end")
        < F.col("billing_period_start")
    )
    | (
        F.col("due_date")
        < F.col("invoice_date")
    )
    | (
        F.col("invoice_date")
        > F.col("snapshot_date")
    )
    | (
        F.datediff(
            F.col("due_date"),
            F.col("invoice_date"),
        )
        != F.col("payment_terms_days")
    )
)

expected_discount_amount = F.round(
    F.col("list_price_amount")
    * F.col("discount_percentage")
    / F.lit(100),
    2,
)

expected_net_subscription_amount = F.round(
    F.col("list_price_amount")
    - expected_discount_amount,
    2,
)

expected_subtotal_amount = F.round(
    expected_net_subscription_amount
    + F.col("overage_amount"),
    2,
)

expected_tax_amount = F.round(
    expected_subtotal_amount
    * F.col("tax_rate")
    / F.lit(100),
    2,
)

expected_invoice_total_amount = F.round(
    expected_subtotal_amount
    + expected_tax_amount,
    2,
)

invalid_invoice_financial_condition = (
    (F.col("list_price_amount") < 0)
    | ~F.col("discount_percentage").between(
        0,
        100,
    )
    | (F.col("overage_amount") < 0)
    | ~F.col("tax_rate").between(0, 100)
    | (
        F.abs(
            F.col("discount_amount")
            - expected_discount_amount
        )
        > F.lit(0.01)
    )
    | (
        F.abs(
            F.col("net_subscription_amount")
            - expected_net_subscription_amount
        )
        > F.lit(0.01)
    )
    | (
        F.abs(
            F.col("subtotal_amount")
            - expected_subtotal_amount
        )
        > F.lit(0.01)
    )
    | (
        F.abs(
            F.col("tax_amount")
            - expected_tax_amount
        )
        > F.lit(0.01)
    )
    | (
        F.abs(
            F.col("invoice_total_amount")
            - expected_invoice_total_amount
        )
        > F.lit(0.01)
    )
)

invalid_balance_reconciliation_condition = (
    F.abs(
        F.col("invoice_total_amount")
        - (
            F.col("amount_paid")
            + F.col("outstanding_amount")
            + F.col("voided_amount")
        )
    )
    > F.lit(0.01)
)

invalid_status_allocation_condition = (
    (
        F.col("invoice_status") == "Paid"
    )
    & (
        (
            F.abs(
                F.col("amount_paid")
                - F.col("invoice_total_amount")
            )
            > F.lit(0.01)
        )
        | (F.col("outstanding_amount") != 0)
        | (F.col("voided_amount") != 0)
    )
) | (
    F.col("invoice_status").isin(
        "Open",
        "Past Due",
    )
    & (
        (F.col("amount_paid") != 0)
        | (
            F.abs(
                F.col("outstanding_amount")
                - F.col("invoice_total_amount")
            )
            > F.lit(0.01)
        )
        | (F.col("voided_amount") != 0)
    )
) | (
    (
        F.col("invoice_status") == "Voided"
    )
    & (
        (F.col("amount_paid") != 0)
        | (F.col("outstanding_amount") != 0)
        | (
            F.abs(
                F.col("voided_amount")
                - F.col("invoice_total_amount")
            )
            > F.lit(0.01)
        )
    )
)

invoice_validation_metrics = (
    silver_invoices_df
    .agg(
        F.count("*").alias(
            "silver_invoice_count"
        ),
        F.countDistinct("invoice_id").alias(
            "distinct_invoice_count"
        ),
        F.sum(
            F.when(
                required_invoice_field_is_missing,
                1,
            ).otherwise(0)
        ).alias("null_required_field_count"),
        F.sum(
            F.when(
                invalid_invoice_status_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_status_count"),
        F.sum(
            F.when(
                invalid_invoice_frequency_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_billing_frequency_count"
        ),
        F.sum(
            F.when(
                invalid_invoice_currency_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_currency_count"),
        F.sum(
            F.when(
                invalid_invoice_operation_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_operation_count"),
        F.sum(
            F.when(
                invalid_invoice_date_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_date_count"),
        F.sum(
            F.when(
                invalid_invoice_financial_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_financial_calculation_count"
        ),
        F.sum(
            F.when(
                invalid_balance_reconciliation_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_balance_reconciliation_count"
        ),
        F.sum(
            F.when(
                invalid_status_allocation_condition,
                1,
            ).otherwise(0)
        ).alias(
            "invalid_status_allocation_count"
        ),
    )
    .first()
    .asDict()
)

duplicate_invoice_count = (
    invoice_validation_metrics[
        "silver_invoice_count"
    ]
    - invoice_validation_metrics[
        "distinct_invoice_count"
    ]
)

excluded_invoice_count = (
    orphan_invoice_states_df.count()
)

ownership_mismatch_count = (
    ownership_mismatch_invoice_states_df.count()
)

for metric_name, metric_value in (
    invoice_validation_metrics.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:,}"
    )

print(
    f"duplicate_invoice_count: "
    f"{duplicate_invoice_count:,}"
)

print(
    f"excluded_orphan_invoice_count: "
    f"{excluded_invoice_count:,}"
)

print(
    f"ownership_mismatch_count: "
    f"{ownership_mismatch_count:,}"
)

assert (
    invoice_validation_metrics[
        "silver_invoice_count"
    ]
    == EXPECTED_SILVER_INVOICE_COUNT
), "Unexpected Silver invoice count."

assert duplicate_invoice_count == 0, (
    "Duplicate invoice IDs detected."
)

assert (
    excluded_invoice_count
    == EXPECTED_EXCLUDED_INVOICE_COUNT
), "Unexpected excluded invoice count."

assert ownership_mismatch_count == 0, (
    "Invoice ownership mismatches detected."
)

for metric_name in [
    "null_required_field_count",
    "invalid_status_count",
    "invalid_billing_frequency_count",
    "invalid_currency_count",
    "invalid_operation_count",
    "invalid_date_count",
    "invalid_financial_calculation_count",
    "invalid_balance_reconciliation_count",
    "invalid_status_allocation_count",
]:
    assert (
        invoice_validation_metrics[
            metric_name
        ]
        == 0
    ), f"Validation failed: {metric_name}"

print(
    "Silver invoice validation completed successfully."
)

display(
    silver_invoices_df
    .groupBy("invoice_status")
    .agg(
        F.count("*").alias("invoice_count"),
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias("invoice_total_amount"),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias("amount_paid"),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias("outstanding_amount"),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias("voided_amount"),
    )
    .orderBy("invoice_status")
)

## 4. Persist the Silver Invoice Table

Persist the validated current-state invoices as a managed Delta table.

The merge uses `invoice_id` as the business key and updates an invoice only when its source record hash changes. New invoices are inserted and records absent from the resolved valid state are removed.

In [0]:
spark.sql(
    """
    CREATE SCHEMA IF NOT EXISTS
    workspace.revenue_leakage_silver
    """
)

silver_invoices_df.createOrReplaceTempView(
    "silver_invoice_updates"
)

if spark.catalog.tableExists(
    SILVER_INVOICES_TABLE
):
    spark.sql(
        f"""
        MERGE INTO
          {SILVER_INVOICES_TABLE} AS target
        USING
          silver_invoice_updates AS source
        ON
          target.invoice_id
          = source.invoice_id

        WHEN MATCHED
          AND target._record_hash
              <> source._record_hash
        THEN
          UPDATE SET *

        WHEN NOT MATCHED THEN
          INSERT *

        WHEN NOT MATCHED BY SOURCE THEN
          DELETE
        """
    )

    write_method = "Delta MERGE"

else:
    (
        silver_invoices_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            SILVER_INVOICES_TABLE
        )
    )

    write_method = (
        "Initial Delta table creation"
    )

saved_silver_invoices_df = spark.table(
    SILVER_INVOICES_TABLE
)

saved_invoice_count = (
    saved_silver_invoices_df.count()
)

saved_distinct_invoice_count = (
    saved_silver_invoices_df
    .select("invoice_id")
    .distinct()
    .count()
)

saved_subscription_ownership_errors = (
    saved_silver_invoices_df.alias("invoice")
    .join(
        subscription_ownership_df.alias(
            "subscription"
        ),
        on="subscription_id",
        how="left",
    )
    .filter(
        F.col(
            "subscription._reference_customer_id"
        ).isNull()
        | (
            F.col("invoice.customer_id")
            != F.col(
                "subscription._reference_customer_id"
            )
        )
    )
    .count()
)

assert (
    saved_invoice_count
    == EXPECTED_SILVER_INVOICE_COUNT
), "Saved Silver invoice count is incorrect."

assert (
    saved_distinct_invoice_count
    == saved_invoice_count
), "Saved table contains duplicate invoices."

assert (
    saved_subscription_ownership_errors
    == 0
), "Saved table contains ownership errors."

print(
    f"Write method: {write_method}"
)

print(
    f"Silver table: "
    f"{SILVER_INVOICES_TABLE}"
)

print(
    f"Saved Silver invoices: "
    f"{saved_invoice_count:,}"
)

print(
    f"Saved distinct invoice IDs: "
    f"{saved_distinct_invoice_count:,}"
)

print(
    f"Saved ownership errors: "
    f"{saved_subscription_ownership_errors:,}"
)

display(
    saved_silver_invoices_df
    .groupBy("invoice_status")
    .agg(
        F.count("*").alias("invoice_count"),
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias("invoice_total_amount"),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias("amount_paid"),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias("outstanding_amount"),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias("voided_amount"),
    )
    .orderBy("invoice_status")
)

## 5. Validate Idempotent Reprocessing

Reapply the same resolved invoice state through the Delta merge.

A successful rerun must produce zero inserts, updates, and deletes while preserving invoice uniqueness, ownership, and financial totals.

In [0]:
rows_before_invoice_rerun = (
    spark.table(
        SILVER_INVOICES_TABLE
    )
    .count()
)

spark.sql(
    f"""
    MERGE INTO
      {SILVER_INVOICES_TABLE} AS target
    USING
      silver_invoice_updates AS source
    ON
      target.invoice_id
      = source.invoice_id

    WHEN MATCHED
      AND target._record_hash
          <> source._record_hash
    THEN
      UPDATE SET *

    WHEN NOT MATCHED THEN
      INSERT *

    WHEN NOT MATCHED BY SOURCE THEN
      DELETE
    """
)

latest_invoice_history_df = (
    spark.sql(
        f"""
        DESCRIBE HISTORY
        {SILVER_INVOICES_TABLE}
        LIMIT 1
        """
    )
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics",
    )
)

latest_invoice_history_row = (
    latest_invoice_history_df.first()
)

invoice_operation_metrics = (
    latest_invoice_history_row[
        "operationMetrics"
    ]
    or {}
)

rows_inserted = int(
    invoice_operation_metrics.get(
        "numTargetRowsInserted",
        "0",
    )
)

rows_updated = int(
    invoice_operation_metrics.get(
        "numTargetRowsUpdated",
        "0",
    )
)

rows_deleted = int(
    invoice_operation_metrics.get(
        "numTargetRowsDeleted",
        "0",
    )
)

invoices_after_rerun_df = spark.table(
    SILVER_INVOICES_TABLE
)

rows_after_invoice_rerun = (
    invoices_after_rerun_df.count()
)

distinct_invoices_after_rerun = (
    invoices_after_rerun_df
    .select("invoice_id")
    .distinct()
    .count()
)

duplicate_invoices_after_rerun = (
    rows_after_invoice_rerun
    - distinct_invoices_after_rerun
)

financial_totals_after_rerun = (
    invoices_after_rerun_df
    .agg(
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias("invoice_total"),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias("paid_total"),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias("outstanding_total"),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias("voided_total"),
    )
    .first()
    .asDict()
)

assert (
    latest_invoice_history_row["operation"]
    == "MERGE"
), "Latest Delta operation was not MERGE."

assert rows_inserted == 0, (
    "Idempotency failed: rows were inserted."
)

assert rows_updated == 0, (
    "Idempotency failed: rows were updated."
)

assert rows_deleted == 0, (
    "Idempotency failed: rows were deleted."
)

assert (
    rows_before_invoice_rerun
    == EXPECTED_SILVER_INVOICE_COUNT
), "Unexpected count before rerun."

assert (
    rows_after_invoice_rerun
    == EXPECTED_SILVER_INVOICE_COUNT
), "Unexpected count after rerun."

assert duplicate_invoices_after_rerun == 0, (
    "Duplicate invoices detected."
)

print(
    f"Rows before rerun: "
    f"{rows_before_invoice_rerun:,}"
)

print(
    f"Rows after rerun: "
    f"{rows_after_invoice_rerun:,}"
)

print(
    f"Rows inserted during rerun: "
    f"{rows_inserted:,}"
)

print(
    f"Rows updated during rerun: "
    f"{rows_updated:,}"
)

print(
    f"Rows deleted during rerun: "
    f"{rows_deleted:,}"
)

print(
    f"Duplicate invoices: "
    f"{duplicate_invoices_after_rerun:,}"
)

for metric_name, metric_value in (
    financial_totals_after_rerun.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:,.2f}"
    )

print(
    "Silver invoice transformation is idempotent."
)

display(
    latest_invoice_history_df
)

## Result

The invoice Silver transformation completed successfully:

- 26,925 Bronze invoice events processed
- 226 invoices without a current Silver subscription excluded
- 26,699 valid current-state invoices persisted
- Zero duplicate invoice identifiers
- Zero subscription ownership mismatches
- Zero domain, date, financial-calculation, or balance-allocation violations
- Invoice total: 4,615,670.55 USD
- Paid total: 3,806,266.54 USD
- Outstanding total: 729,286.01 USD
- Voided total: 80,118.00 USD
- Delta merge rerun produced zero changes
- Target confirmed as a managed Delta table